# LangChain agent with EconDash tools

Plain functions become LangChain tools. A dedicated `langchain-econdash` package is planned;
until then this is all it takes. Needs an LLM key, here `ANTHROPIC_API_KEY`.

In [ ]:
%pip install -q econdash langchain "langchain[anthropic]"

In [ ]:
from econdash import EconDash

ed = EconDash()


def search_indicators(query: str) -> str:
    """Find EconDash indicator slugs by keywords, e.g. 'inflation' or 'government debt'."""
    return "\n".join(
        f"{i.slug}: {i.name_en} [{i.unit}]" for i in ed.search_indicators(query, limit=8)
    )


def get_series(indicator: str, country_iso3: str, start_year: int = 2000) -> str:
    """Yearly values of an indicator (slug) for a country (ISO3 code, e.g. USA, DEU)."""
    s = ed.get_series(indicator, country_iso3, start=start_year)
    rows = "\n".join(f"{p.date}: {p.value:,.4g}" for p in s.data)
    return f"{s.indicator_name_en} ({s.unit}), {country_iso3}\n{rows}"


def cite(indicator: str, country_iso3: str) -> str:
    """Latest value with a citation sentence and source link."""
    c = ed.cite(indicator, country_iso3)
    return f"{c.citation.text}. Link: {c.meta.canonical_url}"

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="anthropic:claude-sonnet-5-5",
    tools=[search_indicators, get_series, cite],
    system_prompt="You answer macroeconomic questions with EconDash data. Always cite the source.",
)
result = agent.invoke(
    {"messages": [{"role": "user", "content": "How did inflation in Germany change since 2019?"}]}
)
print(result["messages"][-1].content)